# Fine-tune reranker on src2 Stage 1 candidates - Kaggle

This notebook trains a cross-encoder reranker for `query -> paper` scoring.

It is intentionally similar in structure to the E5 Kaggle notebook:
- config-driven
- works on `2x T4` with `DataParallel`
- mixed precision training
- saves the best checkpoint by `dev MRR@5`

## Required Kaggle datasets
1. A dataset with Stage 1 exports from `src2`, for example:
   - `/kaggle/input/ct26-src2-stage1-top50/src2_stage1_train_top50.jsonl`
   - `/kaggle/input/ct26-src2-stage1-top50/src2_stage1_dev_top50.jsonl`
2. A base reranker model, or internet enabled. Example model:
   - `BAAI/bge-reranker-v2-m3`
3. Optional: hard negatives dataset for extra training pairs.

## What is `top-50 candidates`?
For each query, `src2` retrieval returns the top 50 papers before reranking.
The reranker then only reorders those 50 papers. It does not search the full collection.

## How to create the export locally
Use the new script:

```powershell
python .\scripts\export_stage1_candidates_src2.py --config .\configs\src2_translated_hybrid.yaml --split train --top-n 50 --output .\exports\src2_stage1_train_top50.jsonl
python .\scripts\export_stage1_candidates_src2.py --config .\configs\src2_translated_hybrid.yaml --split dev --top-n 50 --output .\exports\src2_stage1_dev_top50.jsonl
```

Each row contains:
- `query`
- `true_pubkey`
- `true_text`
- `candidates` with `rank`, `pubkey`, `text`


In [ ]:
# -- 0. GPU check -----------------------------------------------------------
import subprocess
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else 'No GPU found')
except FileNotFoundError:
    print('GPU is not enabled. Turn on Settings -> Accelerator -> GPU T4 x2 if available.')


In [ ]:
# -- 1. Install + restart kernel -------------------------------------------
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'sentence-transformers'])
print('Restarting kernel...')
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


In [ ]:
# -- 2. Config --------------------------------------------------------------
import torch
from pathlib import Path

N_GPUS = torch.cuda.device_count()
print(f'GPUs: {N_GPUS}x  ({[torch.cuda.get_device_name(i) for i in range(N_GPUS)]})')

TRAIN_EXPORT_PATH = Path('/kaggle/input/ct26-src2-stage1-top50/src2_stage1_train_top50.jsonl')
DEV_EXPORT_PATH   = Path('/kaggle/input/ct26-src2-stage1-top50/src2_stage1_dev_top50.jsonl')
HARD_NEG_PATH     = Path('/kaggle/input/ct26-hard-negatives-v2/hard_negatives_v2.jsonl')
OUTPUT_DIR        = Path('/kaggle/working/reranker-finetuned-src2')

# Fill in your model path or Hugging Face model id.
MODEL_NAME = 'BAAI/bge-reranker-v2-m3'

EPOCHS                    = 3
BATCH_SIZE                = 32 if N_GPUS >= 2 else 8
GRAD_ACCUM                = 1 if N_GPUS >= 2 else 4
LR                        = 2e-5
MAX_LENGTH                = 512
LOSS_TYPE                 = 'logsigmoid'   # 'logsigmoid' or 'margin'
MARGIN                    = 1.0
TRAIN_STAGE1_NEGATIVES    = 4
DEV_RERANK_TOP_K          = 5
USE_HARD_NEGATIVES        = True
HARD_NEGATIVES_PER_QUERY  = 2
FOCUS_ON_STAGE1_ERRORS    = False
MAX_GOLD_RANK            = 25
BALANCE_LANGS             = True
DOC_TEXT_MODE             = 'raw'  # 'raw' or 'key_sentences'
KEY_SENT_TOP_K            = 3
KEY_SENT_MAX_SENTENCES    = 8
MAX_TRAIN_QUERIES         = 0
MAX_DEV_QUERIES           = 0
SEED                      = 42
LOG_EVERY                 = 20

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert TRAIN_EXPORT_PATH.exists(), f'Missing train export: {TRAIN_EXPORT_PATH}'
assert DEV_EXPORT_PATH.exists(), f'Missing dev export: {DEV_EXPORT_PATH}'
if USE_HARD_NEGATIVES:
    assert HARD_NEG_PATH.exists(), f'Missing hard negatives: {HARD_NEG_PATH}'
print(f'Train export: {TRAIN_EXPORT_PATH}')
print(f'Dev export:   {DEV_EXPORT_PATH}')
if USE_HARD_NEGATIVES:
    print(f'Hard negs:    {HARD_NEG_PATH}')
print(f'Model:        {MODEL_NAME}')
print(f'Batch size:   {BATCH_SIZE}  (grad_accum={GRAD_ACCUM} -> effective={BATCH_SIZE * GRAD_ACCUM})')
print(f'Loss:         {LOSS_TYPE}')
print(f'Doc mode:     {DOC_TEXT_MODE}')
print(f'Output:       {OUTPUT_DIR}')


In [ ]:
# -- 3. Build reranker dataset ---------------------------------------------
import json, random, re
from collections import Counter, defaultdict
from torch.utils.data import Dataset, DataLoader

random.seed(SEED)
torch.manual_seed(SEED)
device = 'cuda'
PAD_TOKEN_ID = 0


def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]


def maybe_limit(rows, limit, seed):
    if not limit or len(rows) <= limit:
        return rows
    rng = random.Random(seed)
    keep = sorted(rng.sample(range(len(rows)), limit))
    return [rows[i] for i in keep]


def normalize_space(text):
    return re.sub(r'\s+', ' ', text).strip()


def split_sentences(text):
    text = normalize_space(text)
    if not text:
        return []
    pieces = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in pieces if p.strip()]


def build_doc_text(query, text):
    body = normalize_space(text)  # keep 'passage: ' prefix — consistent with inference
    if DOC_TEXT_MODE == 'raw':
        return body
    if DOC_TEXT_MODE != 'key_sentences':
        raise ValueError(f'Unsupported DOC_TEXT_MODE: {DOC_TEXT_MODE}')

    sentences = split_sentences(body)
    if len(sentences) <= KEY_SENT_TOP_K:
        return body

    q_terms = set(re.findall(r'\w+', query.lower()))
    query_has_digits = any(ch.isdigit() for ch in query)
    scored = []
    for idx, sent in enumerate(sentences[:KEY_SENT_MAX_SENTENCES]):
        terms = re.findall(r'\w+', sent.lower())
        overlap = sum(1 for t in terms if t in q_terms)
        digit_bonus = 2 if query_has_digits and any(ch.isdigit() for ch in sent) else 0
        score = overlap + digit_bonus - idx * 0.01
        scored.append((score, idx, sent))

    chosen = sorted(sorted(scored, reverse=True)[:KEY_SENT_TOP_K], key=lambda x: x[1])
    if max(score for score, _, _ in chosen) <= 0:
        chosen = [(0.0, idx, sent) for idx, sent in enumerate(sentences[:KEY_SENT_TOP_K])]
    return ' '.join(sent for _, _, sent in chosen)


def get_true_text(row):
    true_pubkey = row.get('true_pubkey')
    true_text = row.get('true_text', '').strip()
    if true_text:
        return true_text
    for cand in row.get('candidates', []):
        if cand.get('pubkey') == true_pubkey:
            return cand.get('text', '').strip()
    return ''


def get_true_rank(row):
    true_pubkey = row.get('true_pubkey')
    for cand in row.get('candidates', []):
        if cand.get('pubkey') == true_pubkey:
            return int(cand.get('rank', 0) or 0)
    return None


def load_hard_negative_map(path):
    mapping = {}
    if not path.exists():
        return mapping
    with open(path, encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            q = item['anchor'].removeprefix('query: ')
            mapping[q] = list(item.get('negatives', []))  # keep 'passage: ' prefix
    return mapping


def choose_stage1_negatives(row, negatives_per_query):
    candidates = row.get('candidates', [])
    true_pubkey = row.get('true_pubkey')
    true_rank = get_true_rank(row)

    if FOCUS_ON_STAGE1_ERRORS:
        if true_rank is None or true_rank <= 1 or true_rank > MAX_GOLD_RANK:
            return None
    elif true_rank is not None and true_rank > MAX_GOLD_RANK:
        return None

    pool = []
    for cand in candidates:
        if cand.get('pubkey') == true_pubkey:
            continue
        text = cand.get('text', '').strip()
        if not text:
            continue
        rank = int(cand.get('rank', 0) or 0)
        source = 'stage1_top' if rank and rank <= max(1, (true_rank or MAX_GOLD_RANK) - 1) else 'stage1_tail'
        pool.append({'text': text, 'source': source, 'rank': rank})

    selected = []
    seen = set()
    for item in pool:
        key = normalize_space(item['text'])
        if key in seen:
            continue
        seen.add(key)
        selected.append(item)
        if len(selected) >= negatives_per_query:
            break
    return selected


def build_stage1_examples(rows, negatives_per_query, hard_negative_map=None):
    examples = []
    skipped = Counter()
    source_counts = Counter()

    for row in rows:
        query = normalize_space(row['query'])
        true_text = get_true_text(row)
        if not query or not true_text:
            skipped['missing_positive'] += 1
            continue

        negatives = choose_stage1_negatives(row, negatives_per_query)
        if not negatives:
            skipped['no_stage1_signal'] += 1
            continue

        dedup = {normalize_space(item['text']) for item in negatives}
        if hard_negative_map is not None:
            for neg in hard_negative_map.get(query, [])[:HARD_NEGATIVES_PER_QUERY]:
                key = normalize_space(neg)
                if key in dedup:
                    continue
                dedup.add(key)
                negatives.append({'text': neg, 'source': 'hard_dense', 'rank': 999})

        for item in negatives:
            source_counts[item['source']] += 1

        examples.append({
            'query': query,
            'positive': true_text,
            'negatives': negatives,
            'lang': row.get('lang') or 'unk',
            'index': row.get('index'),
            'true_rank': get_true_rank(row),
        })

    return examples, skipped, source_counts


def balance_examples_by_lang(examples, seed):
    grouped = defaultdict(list)
    for ex in examples:
        grouped[ex['lang']].append(ex)
    if len(grouped) <= 1:
        return examples
    min_count = min(len(v) for v in grouped.values())
    rng = random.Random(seed)
    balanced = []
    for lang, items in sorted(grouped.items()):
        if len(items) > min_count:
            items = rng.sample(items, min_count)
        balanced.extend(items)
    rng.shuffle(balanced)
    return balanced


train_rows = maybe_limit(load_jsonl(TRAIN_EXPORT_PATH), MAX_TRAIN_QUERIES, SEED)
dev_rows   = maybe_limit(load_jsonl(DEV_EXPORT_PATH), MAX_DEV_QUERIES, SEED)

hn_map = load_hard_negative_map(HARD_NEG_PATH) if USE_HARD_NEGATIVES else None
train_examples, train_skipped, train_sources = build_stage1_examples(train_rows, TRAIN_STAGE1_NEGATIVES, hn_map)
dev_examples, dev_skipped, dev_sources       = build_stage1_examples(dev_rows, TRAIN_STAGE1_NEGATIVES, None)

if BALANCE_LANGS:
    train_examples = balance_examples_by_lang(train_examples, SEED)

n_train_pairs = sum(len(ex['negatives']) for ex in train_examples)
n_dev_pairs   = sum(len(ex['negatives']) for ex in dev_examples)
print(f'Train queries: {len(train_examples)}  | skipped={dict(train_skipped)} | pairwise pairs={n_train_pairs}')
print(f'Dev queries:   {len(dev_examples)}  | skipped={dict(dev_skipped)} | pairwise pairs={n_dev_pairs}')
print('Train langs:  ', dict(Counter(ex['lang'] for ex in train_examples)))
print('Train negs:   ', dict(train_sources))
print('Dev negs:     ', dict(dev_sources))


class PairwiseDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length):
        self.pos_input_ids = []
        self.pos_attention_mask = []
        self.neg_input_ids = []
        self.neg_attention_mask = []
        self.langs = []
        self.sources = []

        for ex in examples:
            pos_text = build_doc_text(ex['query'], ex['positive'])
            pos_enc = tokenizer(ex['query'], pos_text, padding=False, truncation=True, max_length=max_length)
            for neg in ex['negatives']:
                neg_text = build_doc_text(ex['query'], neg['text'])
                neg_enc = tokenizer(ex['query'], neg_text, padding=False, truncation=True, max_length=max_length)
                self.pos_input_ids.append(pos_enc['input_ids'])
                self.pos_attention_mask.append(pos_enc['attention_mask'])
                self.neg_input_ids.append(neg_enc['input_ids'])
                self.neg_attention_mask.append(neg_enc['attention_mask'])
                self.langs.append(ex['lang'])
                self.sources.append(neg['source'])

    def __len__(self):
        return len(self.neg_input_ids)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.pos_input_ids[idx], dtype=torch.long),
            torch.tensor(self.pos_attention_mask[idx], dtype=torch.long),
            torch.tensor(self.neg_input_ids[idx], dtype=torch.long),
            torch.tensor(self.neg_attention_mask[idx], dtype=torch.long),
            self.langs[idx],
            self.sources[idx],
        )


def collate_pairwise(batch):
    pos_ids, pos_masks, neg_ids, neg_masks, langs, sources = zip(*batch)
    pos_ids = torch.nn.utils.rnn.pad_sequence(pos_ids, batch_first=True, padding_value=PAD_TOKEN_ID)
    pos_masks = torch.nn.utils.rnn.pad_sequence(pos_masks, batch_first=True, padding_value=0)
    neg_ids = torch.nn.utils.rnn.pad_sequence(neg_ids, batch_first=True, padding_value=PAD_TOKEN_ID)
    neg_masks = torch.nn.utils.rnn.pad_sequence(neg_masks, batch_first=True, padding_value=0)
    return pos_ids, pos_masks, neg_ids, neg_masks, list(langs), list(sources)


In [ ]:
# -- 4. Model + dataloader --------------------------------------------------
import time
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm.notebook import tqdm

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.sep_token
PAD_TOKEN_ID = tokenizer.pad_token_id

print(f'Loading model: {MODEL_NAME}')
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(device)
model.config.pad_token_id = tokenizer.pad_token_id
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()

if N_GPUS >= 2:
    model = nn.DataParallel(model)
    print(f'DataParallel: {N_GPUS}x GPU -> batch {BATCH_SIZE} -> {BATCH_SIZE // N_GPUS}/GPU')

torch.backends.cudnn.benchmark = True
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'{n_params:.0f}M parameters | VRAM cuda:0: {torch.cuda.memory_allocated(0)/1e9:.2f} GB')

train_dataset = PairwiseDataset(train_examples, tokenizer, MAX_LENGTH)
print('Pairwise train samples:', len(train_dataset))
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
    collate_fn=collate_pairwise,
)
print(f'{len(train_loader)} steps/epoch')


In [ ]:
# -- 5. Train + evaluate ----------------------------------------------------
def unwrap_model(model):
    base = model.module if isinstance(model, nn.DataParallel) else model
    return getattr(base, '_orig_mod', base)


def score_pairs(queries, docs, batch_size=64):
    scores = []
    model.eval()
    for i in range(0, len(queries), batch_size):
        q = queries[i:i + batch_size]
        d = [build_doc_text(qi, di) for qi, di in zip(q, docs[i:i + batch_size])]
        enc = tokenizer(q, d, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(device)
        with torch.no_grad():
            with torch.amp.autocast('cuda'):
                logits = model(**enc).logits.squeeze(-1)
        scores.extend(logits.float().detach().cpu().tolist())
    return scores


def eval_stage1_rows(rows, top_k=5):
    rr = []
    for row in tqdm(rows, desc='Dev rerank', leave=False):
        query = row['query']
        true_pubkey = row.get('true_pubkey')
        candidates = row.get('candidates', [])[:top_k]  # limit to top_k — matches inference
        if not candidates:
            rr.append(0.0)
            continue

        docs = [cand['text'] for cand in candidates]
        pubkeys = [cand['pubkey'] for cand in candidates]
        queries = [query] * len(docs)
        scores = score_pairs(queries, docs, batch_size=max(32, BATCH_SIZE))
        ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)

        hit = 0.0
        for rank, idx in enumerate(ranked[:top_k], start=1):
            if pubkeys[idx] == true_pubkey:
                hit = 1.0 / rank
                break
        rr.append(hit)

    return sum(rr) / len(rr)


def eval_stage1_rows_by_lang(rows, top_k=5):
    grouped = defaultdict(list)
    for row in rows:
        grouped[row.get('lang', 'unk')].append(row)
    metrics = {}
    for lang in sorted(grouped):
        metrics[lang] = eval_stage1_rows(grouped[lang], top_k=top_k)
    return metrics


def pairwise_loss(pos_scores, neg_scores):
    if LOSS_TYPE == 'margin':
        target = torch.ones_like(pos_scores)
        return F.margin_ranking_loss(pos_scores, neg_scores, target, margin=MARGIN)
    if LOSS_TYPE == 'logsigmoid':
        return -F.logsigmoid(pos_scores - neg_scores).mean()
    raise ValueError(f'Unsupported LOSS_TYPE: {LOSS_TYPE}')


total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
warmup_steps = max(1, int(total_steps * 0.1))

try:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01, fused=True)
    print('fused AdamW')
except Exception:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda s: s / max(1, warmup_steps) if s < warmup_steps else max(0.0, 1.0 - (s - warmup_steps) / max(1, total_steps - warmup_steps))
)
scaler = torch.amp.GradScaler('cuda')

best_dev_mrr5 = -1.0
best_dev_by_lang = {}
epoch_history = []
best_path = OUTPUT_DIR / 'best'
last_path = OUTPUT_DIR / 'last'
t0 = time.time()

print(f'Training: {EPOCHS} epochs x {len(train_loader)} steps = {total_steps} total')
print(f'Warmup:   {warmup_steps} steps')

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')

    for step, (pos_ids, pos_masks, neg_ids, neg_masks, langs, sources) in enumerate(pbar):
        pos_ids = pos_ids.to(device, non_blocking=True)
        pos_masks = pos_masks.to(device, non_blocking=True)
        neg_ids = neg_ids.to(device, non_blocking=True)
        neg_masks = neg_masks.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            pos_scores = model(input_ids=pos_ids, attention_mask=pos_masks).logits.squeeze(-1)
            neg_scores = model(input_ids=neg_ids, attention_mask=neg_masks).logits.squeeze(-1)
            loss = pairwise_loss(pos_scores, neg_scores)

        loss = loss / GRAD_ACCUM
        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        epoch_loss += loss.item() * GRAD_ACCUM
        if (step + 1) % LOG_EVERY == 0:
            pbar.set_postfix(loss=f'{loss.item() * GRAD_ACCUM:.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}', src=sources[0], lang=langs[0])

    dev_mrr5 = eval_stage1_rows(dev_rows, top_k=DEV_RERANK_TOP_K)
    dev_by_lang = eval_stage1_rows_by_lang(dev_rows, top_k=DEV_RERANK_TOP_K)
    elapsed = (time.time() - t0) / 60
    print(f'Epoch {epoch}: loss={epoch_loss / len(train_loader):.4f} | dev_mrr@5={dev_mrr5:.4f} | {elapsed:.1f} min')
    print('  Per-lang MRR@5:', {k: round(v, 4) for k, v in dev_by_lang.items()})
    epoch_history.append({
        'epoch': epoch,
        'loss': epoch_loss / len(train_loader),
        'dev_mrr@5': dev_mrr5,
        'dev_mrr@5_by_lang': dev_by_lang,
    })

    raw_model = unwrap_model(model)
    epoch_path = OUTPUT_DIR / f'epoch-{epoch}'
    raw_model.save_pretrained(str(epoch_path))
    tokenizer.save_pretrained(str(epoch_path))

    raw_model.save_pretrained(str(last_path))
    tokenizer.save_pretrained(str(last_path))

    if dev_mrr5 > best_dev_mrr5:
        best_dev_mrr5 = dev_mrr5
        best_dev_by_lang = dict(dev_by_lang)
        raw_model.save_pretrained(str(best_path))
        tokenizer.save_pretrained(str(best_path))
        print(f'  New best -> {best_path}')

metrics = {
    'best_dev_mrr@5': best_dev_mrr5,
    'best_dev_mrr@5_by_lang': best_dev_by_lang,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'grad_accum': GRAD_ACCUM,
    'lr': LR,
    'max_length': MAX_LENGTH,
    'loss_type': LOSS_TYPE,
    'doc_text_mode': DOC_TEXT_MODE,
    'train_stage1_negatives': TRAIN_STAGE1_NEGATIVES,
    'use_hard_negatives': USE_HARD_NEGATIVES,
    'focus_on_stage1_errors': FOCUS_ON_STAGE1_ERRORS,
    'balance_langs': BALANCE_LANGS,
    'train_queries': len(train_examples),
    'pairwise_train_samples': len(train_dataset),
    'epoch_history': epoch_history,
}
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(f'\nDone in {(time.time() - t0) / 60:.1f} min')
print(f'Best dev MRR@5: {best_dev_mrr5:.4f}')
print('Best per-lang MRR@5:', {k: round(v, 4) for k, v in best_dev_by_lang.items()})


In [ ]:
# -- 6. Output --------------------------------------------------------------
total_gb = sum(f.stat().st_size for f in OUTPUT_DIR.rglob('*') if f.is_file()) / 1e9
print(f'Output saved: {OUTPUT_DIR} ({total_gb:.2f} GB)')
print('Best checkpoint: ', OUTPUT_DIR / 'best')
print('Last checkpoint: ', OUTPUT_DIR / 'last')
print('')
print('Use in src2 config:')
print(f'  reranker_model: {OUTPUT_DIR / "best"}')
print('')
print('Recommended next step: publish OUTPUT_DIR as a Kaggle dataset from the Output tab.')
